# P22 review round 6 controls: bias or prompt obedience?
G3s-pl2: heavy placebo, a contentful non-feature instruction cut (TF03 model family, TF05 'fill missing with 0'). G3o-d: the P14 open prompt plus the data profile, so the guide may depart from the wording and has data to judge. Predictions: `quick_runs/predictions/P22.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T14, "G3n")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T14, "G3n-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3n                 5                  2          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3n-p,20,20,10,9,5,0,0,4


In [5]:
T18 = [json.loads(l) for l in open(REPO / "quick_runs/results/P18/guide_trace.jsonl")]
T19 = [json.loads(l) for l in open(REPO / "quick_runs/results/P19/guide_trace.jsonl")]
T22 = [json.loads(l) for l in open(REPO / "quick_runs/results/P22/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T22))
# Heavy placebo vs feature cut vs light placebo, fix lessons shown per task (of 5; cut/placebo also have P19's 10 more)
rows = []
for task, ids in {k: FIX[k] for k in ("sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage")}.items():
    row = {"task": task[:7]}
    row["G3 (none cut)"] = sum(bool(set(t["shown"]) & ids) for t in T13 if t["arm"] == "G3" and t["task"] == task)
    row["G3s feature cut"] = sum(bool(set(t["shown"]) & ids) for t in T18 + T19 if t["arm"] == "G3s" and t["task"] == task)
    row["G3s-pl light"] = sum(bool(set(t["shown"]) & ids) for t in T18 + T19 if t["arm"] == "G3s-pl" and t["task"] == task)
    row["G3s-pl2 heavy"] = sum(bool(set(t["shown"]) & ids) for t in T22 if t["arm"] == "G3s-pl2" and t["task"] == task)
    rows.append(row)
heavy = pd.DataFrame(rows); print(heavy.to_string()); heavy

Counter({'G3o-d': 30, 'G3s-pl2': 10})
      task  G3 (none cut)  G3s feature cut  G3s-pl light  G3s-pl2 heavy
0  sf_tf03              1               15             1              0
1  sf_tf05              0               14             4              2


,task,G3 (none cut),G3s feature cut,G3s-pl light,G3s-pl2 heavy
0,sf_tf03,1,15,1,0
1,sf_tf05,0,14,4,2


In [6]:
# G3o-d (open prompt + profile): TF03/TF05 fix and keep-column_x shown, vs G3n (open, no data) and G3d (profile)
def fx(trs, arm, task, ids): return sum(bool(set(t["shown"]) & ids) for t in trs if t["arm"] == arm and t["task"] == task)
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T16 = [json.loads(l) for l in open(REPO / "quick_runs/results/P16/guide_trace.jsonl")]
rows = []
for task in ("sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"):
    rows.append({"task": task[:7], "arm": "G3n open", "fix": fx(T14, "G3n", task, FIX[task]), "keepx": fx(T14, "G3n", task, CONTRA3) if task.endswith("causation") else 0})
    rows.append({"task": task[:7], "arm": "G3d profile", "fix": fx(T16, "G3d", task, FIX[task]), "keepx": fx(T16, "G3d", task, CONTRA3) if task.endswith("causation") else 0})
    rows.append({"task": task[:7], "arm": "G3o-d open+prof", "fix": fx(T22, "G3o-d", task, FIX[task]), "keepx": fx(T22, "G3o-d", task, CONTRA3) if task.endswith("causation") else 0})
od = pd.DataFrame(rows); print(od.to_string()); od

      task              arm  fix  keepx
0  sf_tf03         G3n open    0      5
1  sf_tf03      G3d profile    5      0
2  sf_tf03  G3o-d open+prof    5      0
3  sf_tf05         G3n open    2      0
4  sf_tf05      G3d profile    0      0
5  sf_tf05  G3o-d open+prof    0      0


,task,arm,fix,keepx
0,sf_tf03,G3n open,0,5
1,sf_tf03,G3d profile,5,0
2,sf_tf03,G3o-d open+prof,5,0
3,sf_tf05,G3n open,2,0
4,sf_tf05,G3d profile,0,0
5,sf_tf05,G3o-d open+prof,0,0


In [7]:
# Reasons: do G3s-pl2 and G3o-d cite the instructions when rejecting fix lessons? (pre-set regex + contradict rule)
NOC = r"task (says|asks|states|requires|instructs)|instruction|use all|all available"
TASKWORD = r"prompt|task|requirement|constraint"
def cites_instr(reason):
    if not re.search(INSTR, reason, re.I): return False
    return bool(re.search(NOC, reason, re.I) or re.search(TASKWORD, reason, re.I))
rows = []
for t in T22:
    for r in t.get("rounds", []):
        for j in r.get("judgements") or []:
            try: i = int(j.get("id"))
            except (TypeError, ValueError): continue
            if i in FIX.get(t["task"], set()) and i in r["candidates"]:
                rows.append({"arm": t["arm"], "task": t["task"][:7], "useful": j.get("useful") is True,
                             "instr": cites_instr(str(j.get("reason", ""))), "reason": str(j.get("reason", ""))})
jr = pd.DataFrame(rows)
print(jr[~jr.useful].groupby("arm").instr.agg(["sum", "count", "mean"]).round(2).to_string())
for _, r in jr.groupby(["arm", "task", "useful"]).head(1).iterrows(): print(f"{r.arm} {r.task} useful={r.useful}: {r.reason[:200]}")

         sum  count  mean
arm                      
G3o-d     15     32  0.47
G3s-pl2   35     47  0.74
G3s-pl2 sf_tf03 useful=False: This note suggests excluding numerical features based on causal reasoning. However, the task provides a specific instruction to 'Keep only numerical feature columns', and over-filtering might violate 
G3s-pl2 sf_tf05 useful=False: The prompt explicitly instructs the agent to 'Use all available feature columns'. While these features might be leakage in a real-world scenario, the agent must follow the explicit prompt constraint f
G3s-pl2 sf_tf05 useful=True: This note identifies specific leakage features ('majorcards', 'expenditure') in the credit card dataset and explicitly warns the agent to remove them even if the prompt says to 'use all available feat
G3o-d sf_tf03 useful=True: The filename contains 'causal', and the data contains a blatant spurious correlation ('column_x'). This note encourages the agent to reason about the features rather than blindl

In [8]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P12_C1", "P13_E", "P13_G3", "P14_G3n", "P16_G3d", "P22_G3od"]
NAME = dict(zip(ARMS, ["A0 none", "C1 own", "E emb", "G3", "G3n open", "G3d profile", "G3o-d open+prof"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G3': (30, {'final_answer': 29, 'max_steps': 1}), 'P14_G3n': (30, {'final_answer': 30}), 'P16_G3d': (30, {'final_answer': 28, 'max_steps': 2}), 'P22_G3od': (30, {'final_answer': 29, 'max_steps': 1})}
pooled: {'A0 none': 0.075, 'C1 own': 0.75, 'E emb': 0.475, 'G3': 0.5, 'G3n open': 0.45, 'G3d profile': 0.6, 'G3o-d open+prof': 0.675}


,task,check,A0 none,C1 own,E emb,G3,G3n open,G3d profile,G3o-d open+prof
0,sf_tf01,checks_class_distribution,0,5,5,4,3,2,4
1,sf_tf01,uses_class_weight,0,5,5,5,5,5,5
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,5,5,5,5,5
4,sf_tf02,handles_unit_suffix_in_height,0,4,4,4,5,3,4
5,sf_tf03,column_x_included,0,5,0,1,0,5,5
6,sf_tf03,applies_causal_reasoning,0,5,0,1,0,4,4
7,sf_tf05,used_leakage_features,0,1,0,0,0,0,0


In [9]:
# Harm and exact tests
from scipy.stats import fisher_exact
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
passed = (pooled * 40).round().astype(int)
fp = lambda a, b: round(fisher_exact([[a, 40 - a], [b, 40 - b]])[1], 4)
h = heavy.set_index("task")
# feature cut has 15 runs/task (P18+P19); heavy placebo has 5 (P22)
f03, hv03 = int(h.loc["sf_tf03","G3s feature cut"]), int(h.loc["sf_tf03","G3s-pl2 heavy"])
fisher = {"TF03 fix G3s(15) vs G3s-pl2(5)": round(fisher_exact([[f03, 15-f03],[hv03, 5-hv03]])[1],4),
          "G3o-d vs G3d pooled": fp(passed["G3o-d open+prof"], passed["G3d profile"]),
          "G3o-d vs G3n pooled": fp(passed["G3o-d open+prof"], passed["G3n open"])}
print(fisher); scores

{'TF03 fix G3s(15) vs G3s-pl2(5)': np.float64(0.0001), 'G3o-d vs G3d pooled': np.float64(0.6422), 'G3o-d vs G3n pooled': np.float64(0.0707)}


arm,A0 none,C1 own,E emb,G3,G3d profile,G3n open,G3o-d open+prof
task,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.765000e-01,7.713000e-01,7.715000e-01,7.658000e-01,7.706000e-01,7.708000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.650909e+06,4.650909e+06,4.650909e+06,4.655340e+06,4.653567e+06,4.650909e+06
sf_tf03_correlation_causation,2.769462e+07,4.755266e+06,2.769464e+07,2.310314e+07,4.759155e+06,2.769458e+07,4.769609e+06
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
sf_tf05_credit_card_leakage,9.659000e-01,8.220000e-01,9.144000e-01,9.659000e-01,9.659000e-01,9.280000e-01,9.659000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,9.806000e-01,9.796000e-01,9.803000e-01,9.777000e-01,9.792000e-01,9.797000e-01


In [10]:
summary = {"heavy_placebo": heavy.to_dict("records"), "open_profile": od.to_dict("records"),
  "reasons_instr": jr[~jr.useful].groupby("arm").instr.agg(["sum", "count"]).reset_index().to_dict("records"),
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "fisher_p": fisher,
  "score_mean": scores.to_dict("index")}
json.dump(summary, open(REPO / "quick_runs/results/P22/summary.json", "w"), indent=1, default=str)
jr.to_csv(REPO / "quick_runs/results/P22/fix_reasons.csv", index=False)
"done" 

'done'